In [1]:
!python --version

Python 3.13.0


In [2]:
!pip install langchain langchain-community langchain-core
!pip install faiss-cpu
!pip install -U langchain-core langchain-community langchain-text-splitters langchain-ollama faiss-cpu 
!pip install gradio --user
!pip install langchain-community langchain-openai
!pip install google-generativeai


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.22
    Uninstalling langchain-core-1.2.22:
      Successfully uninstalled langchain-core-1.2.22



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   -


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\Shashwati\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "C:\Users\Shashwati\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ~~~~~~~~~~~~~^^^^^
  File "C:\Users\Shashwati\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ~~~~~~~~~~~~~^^^^^
  File "C:\Users\Shashwati\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
                  ~

In [3]:
import faiss
print(faiss.__version__)

1.13.2


In [4]:
!ollama pull mxbai-embed-large
# !ollama pull codellama:7b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 819c2adf5ce6: 100% ▕██████████████████▏ 669 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling b837481ff855: 100% ▕██████████████████▏   16 B                         
pulling 38badd946f91: 100% ▕██████████████████▏  408 B                         
verifying sha256 digest 
writing manifest 
success 


In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import Language
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.embeddings import OllamaEmbeddings
import os
import warnings
import requests
import json
warnings.filterwarnings('ignore')

# Gemini API Configuration
API_KEY = os.getenv("GEMINI_API_KEY", "Insert your API KEY")  # Set your Gemini API key here or use environment variable


In [22]:
import os
from langchain_core.documents import Document

def code_reader_func(root_dir):
    code_docs = []
    file_index = 0

    for subdir, _, files in os.walk(root_dir):
        for file in files:
            if file.endswith(".java"):
                file_path = os.path.join(subdir, file)
                try:
                    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                        content = f.read().strip()

                    # skip empty files
                    if not content:
                        continue

                    code_docs.append(
                        Document(
                            page_content=content,
                            metadata={
                                "filename": file_path,
                                "file_index": file_index
                            }
                        )
                    )
                    file_index += 1

                except Exception as e:
                    print(f"Skipped: {file_path} → {e}")

    return code_docs


In [9]:
code_doc = code_reader_func(
    r"C:\Users\Shashwati\OneDrive\Documents\Capstone_Project"
)

print("Files loaded:", len(code_doc))


Files loaded: 93


In [10]:
from langchain_community.vectorstores.faiss import FAISS
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
from langchain_core.documents import Document
from langchain_core.callbacks import BaseCallbackHandler


# Split code into chunks
text_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.JAVA,
    chunk_size=2000,
    chunk_overlap=200
)

texts = text_splitter.split_documents(code_doc)
print("Chunks created:", len(texts))


embeddings = OllamaEmbeddings(model="mxbai-embed-large")


Chunks created: 159


In [ ]:
# from langchain.vectorstores import FAISS
# from langchain_community.embeddings import OllamaEmbeddings
# from langchain.text_splitter import RecursiveCharacterTextSplitter, Language
# from langchain_core.documents import Document

# code_doc = [
#     Document(page_content=java_code_string, metadata={"source": "code.java"})
# ]

# text_splitter = RecursiveCharacterTextSplitter.from_language(
#     language=Language.JAVA,
#     chunk_size=2000,
#     chunk_overlap=200
# )

# texts = text_splitter.split_documents(code_doc)

# if not texts:
#     raise ValueError("Text splitting failed")

# embeddings = OllamaEmbeddings(model="mxbai-embed-large")
# vectorstore = FAISS.from_documents(texts, embeddings)


In [ ]:
# !pip install faiss-cpu==1.7.4


In [11]:
import os

if os.path.exists("faiss_index_folder/index.faiss"):
   vector_store = FAISS.load_local("faiss_index_folder", embeddings, allow_dangerous_deserialization=True)
else:
    vector_store = FAISS.from_documents(texts, embeddings)
    vector_store.save_local("faiss_index_folder")

retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})

In [ ]:
# from langchain.vectorstores import Chroma

# vector_store = Chroma.from_documents(
#     documents=texts,
#     embedding=embeddings,
#     persist_directory="./chroma_db"
# )


In [12]:
prompt_RAG = """
You are an expert Java developer and code assistant. Given the context of Java code files from a software project and a user's query, respond precisely and helpfully. Follow these instructions:

1. Use the provided context to understand the code structure, classes, methods, and dependencies.
2. If the query asks for debugging, identify potential issues or errors and suggest correct, syntactically valid Java code.
3. If the query asks for explanation, describe the relevant code behavior and logic clearly.
4. Ensure all suggested Java code is syntactically correct.
5. Include required import statements if necessary.
6. **When suggesting code, enclose it within Markdown code fences using ```java and ``` for proper formatting.**
7. Answer concisely and professionally.

Question:
{question}

Context:
{context}

Helpful Response:
"""
prompt_RAG_template = PromptTemplate(
    template=prompt_RAG, input_variables=["context", "question"]
)


In [13]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [14]:
def generate_response(message):
    """
    Generate a response using RAG (Retrieval-Augmented Generation) with Gemini API.
    
    Args:
        message: User's question or query
    
    Returns:
        str: Response from the Gemini API
    """
    import requests
    import json
    
    try:
        context_docs = retriever.invoke(message)
        
        # Format context from retrieved documents
        if context_docs:
            context = "\n\n".join([doc.page_content for doc in context_docs])
        else:
            context = "No relevant code context found."
        
        # Prepare the prompt with context
        prompt = prompt_RAG_template.format(context=context, question=message)
        
        # Call Gemini API
        url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={API_KEY}"
        
        headers = {
            "Content-Type": "application/json"
        }
        
        data = {
            "contents": [
                {
                    "parts": [
                        {
                            "text": prompt
                        }
                    ]
                }
            ]
        }
        
        response = requests.post(url, headers=headers, json=data)
        
        # Extract and return the response text
        if response.status_code == 200:
            result = response.json()
            output = result["candidates"][0]["content"]["parts"][0]["text"]
            return output
        else:
            return f"Error: {response.status_code} - {response.text}"
    
    except Exception as e:
        return f"An error occurred: {str(e)}"


In [ ]:
# import os

# query = input("Paste the query or error message: ")

# def is_codebase_query(query):
#     keywords = ["class", "method", "debug", "fix", "error", "issue", "dependency", "compile"]
#     return any(word in query.lower() for word in keywords)


# output_dir = "output"
# os.makedirs(output_dir, exist_ok=True)
# filename = "result5.txt"
# file_path = os.path.join(output_dir, filename)

# try:
#     if is_codebase_query(query):
#         results = qa_chain({"query": query})
#         response_text = results["result"]
#     else:
#         prompt_without_context = f"""
#         You are an expert Java developer and code assistant. 
#         Answer this general programming question accurately and concisely.

#         Question: {query}

#         Helpful Response:
#         """
#         response_text = code_llm(prompt_without_context)  # this is a string

#     print("📌 Answer:\n", response_text)

#     with open(file_path, "w", encoding="utf-8") as f:
#         f.write("Query: " + query + "\n\n")
#         f.write(response_text)

#     print(f"\n✅ Result successfully saved in {file_path}")

# except Exception as e:
#     print("\n⚠️ Error occurred:", e)


In [17]:
def chat_copilot(message, history):
    response = generate_response(message)
    # Append in the format expected by gr.Chatbot in newer versions
    history.append({
        "role": "user",
        "content": message
    })
    history.append({
        "role": "assistant",
        "content": response
    })
    return history, history

In [ ]:
import gradio as gr

with gr.Blocks(css="""
body {
    background-color: #fff;
    display: flex;
    justify-content: center;
}
#main-container {
    max-width: 700px;
    width: 100%;
    margin: 0px auto;
    padding: 20px;
    background-color: white;
    border-radius: 12px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.1);
}
#chat-area {
    height: 500px;
    overflow-y: auto;
    border: 1px solid #ddd;
    border-radius: 10px;
    padding: 10px;
}
#input-row {
    display: flex;
    align-items: center;
    margin-top: 15px;
}
#message-box {
    flex-grow: 8;
    height: 48px;
    font-size: 16px;
}
#send-btn {
    flex-grow: 0;
    height: 48px;
    margin-left: 10px;
    font-size: 16px;
    border-radius: 8px;
    background-color: #007bff;
    color: white;
    border: none;
}
#send-btn:hover {
    background-color: #0056b3;
}
""") as demo:

    with gr.Column(elem_id="main-container"):
        chatbot = gr.Chatbot(elem_id="chat-area", type="messages")
        state = gr.State([])

        with gr.Row(elem_id="input-row"):
            msg = gr.Textbox(
                show_label=False,
                placeholder="Type your error or question...",
                elem_id="message-box"
            )
            send_btn = gr.Button("Submit", elem_id="send-btn")

        def submit_message(message, history):
            history, new_state = chat_copilot(message, history)
            return history, new_state, ""

        send_btn.click(
            submit_message,
            inputs=[msg, state],
            outputs=[chatbot, state, msg]
        )

        msg.submit(
            submit_message,
            inputs=[msg, state],
            outputs=[chatbot, state, msg]
        )

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
